# Week2_ex1 — Off-nominal tap of a step-up transformer

Reproduces Example 3.12 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a 13.8/345 kV transformer feeding a 500 MW + 100 Mvar load, solved with the tap at 1.00 and at 1.05.

![PowerWorld one-line](Images/Week2_ex1_result_image.png)

In [1]:
# 1. Case data and per-unit conversion
import numpy as np
import pandas as pd
import pandapower as pp

S_BASE = 500.0      # MVA, Simulator Options > Power Flow Solution > General (not 100 for this case)
V_A, V_B = 13.8, 345.0
X_PU = 0.05         # series reactance on the 500 MVA system base, R = 0
TAP_STEP = 0.00625  # pu per step, +/-16 steps -> 0.90 to 1.10
MVA_TOL = 0.5       # PowerWorld MVA convergence tolerance, used for every MW/Mvar check

pd.DataFrame({
    "Quantity": ["System base", "Bus A (slack)", "Bus B (load)", "Transformer X", "Tap step", "MVA convergence tolerance"],
    "PowerWorld": ["500 MVA", "13.8 kV", "345 kV", "0.05 pu, R = 0", "0.00625 pu", "0.5 MVA"],
    "pandapower": ["sn_mva=500", "vn_kv=13.8", "vn_kv=345", "vk_percent=5.0, vkr_percent=0",
                   "tap_step_percent=0.625", "assert tolerance for MW/Mvar"],
})

,Quantity,PowerWorld,pandapower
0,System base,500 MVA,sn_mva=500
1,Bus A (slack),13.8 kV,vn_kv=13.8
2,Bus B (load),345 kV,vn_kv=345
3,Transformer X,"0.05 pu, R = 0","vk_percent=5.0, vkr_percent=0"
4,Tap step,0.00625 pu,tap_step_percent=0.625
5,MVA convergence tolerance,0.5 MVA,assert tolerance for MW/Mvar


In [2]:
# 2. Build and solve at tap 1.00 and 1.05
def build_case(tap_ratio):
    net = pp.create_empty_network(f_hz=60, sn_mva=S_BASE)
    bus_a = pp.create_bus(net, vn_kv=V_A, name="Bus A")
    bus_b = pp.create_bus(net, vn_kv=V_B, name="Bus B")
    pp.create_ext_grid(net, bus_a, vm_pu=1.0, name="Slack")
    pp.create_load(net, bus_b, p_mw=500.0, q_mvar=100.0, name="Load")
    # PowerWorld keeps the off-nominal ratio on the From bus (Bus B), and the impedance on the
    # untapped side, so the tap sits on the 345 kV winding.
    pp.create_transformer_from_parameters(
        net, hv_bus=bus_b, lv_bus=bus_a, sn_mva=S_BASE, vn_hv_kv=V_B, vn_lv_kv=V_A,
        vk_percent=X_PU * 100, vkr_percent=0.0, pfe_kw=0.0, i0_percent=0.0,
        tap_side="hv", tap_neutral=0, tap_min=-16, tap_max=16,
        tap_step_percent=TAP_STEP * 100, tap_pos=round((tap_ratio - 1) / TAP_STEP),
        tap_changer_type="Ratio", name="T1")
    pp.runpp(net, numba=False)
    return net


net_100 = build_case(1.00)
net_105 = build_case(1.05)

pd.DataFrame({
    "Tap ratio": [1.00, 1.05],
    "tap_pos": [int(net_100.trafo.tap_pos.iat[0]), int(net_105.trafo.tap_pos.iat[0])],
    "Bus B voltage [pu]": [net_100.res_bus.vm_pu.iat[1], net_105.res_bus.vm_pu.iat[1]],
    "Bus B angle [deg]": [net_100.res_bus.va_degree.iat[1], net_105.res_bus.va_degree.iat[1]],
    "Slack P [MW]": [net_100.res_ext_grid.p_mw.iat[0], net_105.res_ext_grid.p_mw.iat[0]],
    "Slack Q [Mvar]": [net_100.res_ext_grid.q_mvar.iat[0], net_105.res_ext_grid.q_mvar.iat[0]],
}).round(5)

,Tap ratio,tap_pos,Bus B voltage [pu],Bus B angle [deg],Slack P [MW],Slack Q [Mvar]
0,1.00,0,0.98860,-2.89905,500.0,126.60283
1,1.05,8,1.03804,-2.89905,500.0,126.60283


In [3]:
# 3. Compare against PowerWorld
def compare(rows):
    """Build a PowerWorld vs pandapower table and fail loudly on any miss."""
    df = pd.DataFrame(rows, columns=["Case", "Quantity", "PowerWorld", "pandapower", "Tolerance", "Source"])
    df["Abs error"] = (df["pandapower"] - df["PowerWorld"]).abs()
    bad = df[df["Abs error"] > df["Tolerance"]]
    assert bad.empty, bad
    return df.round({"pandapower": 5, "Abs error": 5})


# PowerWorld values read once from PowerWorld Simulator (Run Mode, Model Explorer > Network)
rows = []
for tap, net, pw in [
    (1.00, net_100, {"v_kv": (341.069, 5e-4, "Buses, Volt (kV)"), "ang": (-2.90, 5e-3, "Buses, Angle (Deg)"),
                     "p": (499.99, MVA_TOL, "Generators, Gen MW"), "q": (126.60, MVA_TOL, "Generators, Gen Mvar")}),
    (1.05, net_105, {"v_kv": (358.12, 5e-3, "One-line, Bus B kV"), "ang": None,
                     "p": (500.02, MVA_TOL, "Generators, Gen MW"), "q": (126.60, MVA_TOL, "Generators, Gen Mvar")}),
]:
    case = f"tap {tap:.2f}"
    v, t, s = pw["v_kv"]
    rows.append([case, "Bus B voltage [kV]", v, net.res_bus.vm_pu.iat[1] * V_B, t, s])
    if pw["ang"]:
        v, t, s = pw["ang"]
        rows.append([case, "Bus B angle [deg]", v, net.res_bus.va_degree.iat[1], t, s])
    v, t, s = pw["p"]
    rows.append([case, "Slack P [MW]", v, net.res_ext_grid.p_mw.iat[0], t, s])
    v, t, s = pw["q"]
    rows.append([case, "Slack Q [Mvar]", v, net.res_ext_grid.q_mvar.iat[0], t, s])

compare(rows)

,Case,Quantity,PowerWorld,pandapower,Tolerance,Source,Abs error
0,tap 1.00,Bus B voltage [kV],341.069,341.06870,0.0005,"Buses, Volt (kV)",0.00030
1,tap 1.00,Bus B angle [deg],-2.900,-2.89905,0.0050,"Buses, Angle (Deg)",0.00095
2,tap 1.00,Slack P [MW],499.990,500.00000,0.5000,"Generators, Gen MW",0.01000
3,tap 1.00,Slack Q [Mvar],126.600,126.60283,0.5000,"Generators, Gen Mvar",0.00283
4,tap 1.05,Bus B voltage [kV],358.120,358.12214,0.0050,"One-line, Bus B kV",0.00214
5,tap 1.05,Slack P [MW],500.020,500.00000,0.5000,"Generators, Gen MW",0.02000
6,tap 1.05,Slack Q [Mvar],126.600,126.60283,0.5000,"Generators, Gen Mvar",0.00283


In [4]:
# 4. What the tap actually changes
# With the impedance on the untapped side, the tap only rescales Bus B voltage:
# V_B(1.05) / V_B(1.00) should be exactly 1.05 and the reactive loss should not move.
v_ratio = net_105.res_bus.vm_pu.iat[1] / net_100.res_bus.vm_pu.iat[1]
dq = round(net_105.res_ext_grid.q_mvar.iat[0] - net_100.res_ext_grid.q_mvar.iat[0], 9) + 0.0
angle_shift = net_105.res_bus.va_degree.iat[1] - net_100.res_bus.va_degree.iat[1]
assert abs(v_ratio - 1.05) < 1e-9 and abs(dq) < 1e-6 and abs(angle_shift) < 1e-9

pd.DataFrame({
    "Check": ["V_B ratio (tap 1.05 / tap 1.00)", "Change in slack Q [Mvar]", "Change in Bus B angle [deg]",
              "Transformer Q loss [Mvar]"],
    "Value": [v_ratio, dq, angle_shift, net_100.res_trafo.ql_mvar.iat[0]],
    "PowerWorld observation": ["1.03804 / 0.98860 = 1.0500", "126.60 -> 126.60", "not shown", "126.60 - 100 = 26.60"],
}).round({"Value": 6})

,Check,Value,PowerWorld observation
0,V_B ratio (tap 1.05 / tap 1.00),1.050000,1.03804 / 0.98860 = 1.0500
1,Change in slack Q [Mvar],-0.000000,126.60 -> 126.60
2,Change in Bus B angle [deg],0.000000,not shown
3,Transformer Q loss [Mvar],26.602828,126.60 - 100 = 26.60
